# Mini LLM: πλήρως σχολιασμένη έκδοση

Αυτό το notebook είναι η εκτελέσιμη υλοποίηση του mini Transformer του Κεφαλαίου 9.

Στόχος των σχολίων δεν είναι να αλλάξουν το πείραμα, αλλά να κάνουν **ορατή τη ροή των δεδομένων**: από τους ελληνικούς χαρακτήρες, στα token IDs και στα batches, έπειτα στα embeddings και στο causal self-attention, στο loss και το backpropagation, και τέλος στην autoregressive παραγωγή χαρακτήρων.

> **Σημείωση:** Το μοντέλο είναι σκόπιμα μικρό και character-level. Είναι εκπαιδευτικό μοντέλο, όχι προσπάθεια παραγωγής ενός σύγχρονου production LLM.


In [ ]:
!pip install torch


In [ ]:
import os
# Αποφυγή σύγκρουσης των βιβλιοθηκών OpenMP στο Mac.
# Η μεταβλητή αυτή είναι workaround για περιβάλλοντα όπου φορτώνονται δύο OpenMP runtimes.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# Περιορίζουμε την PyTorch σε ένα CPU thread. Στο αρχικό πείραμα αυτό χρησιμοποιήθηκε
# για σταθερότερη εκτέλεση σε macOS/Jupyter. Δεν είναι απαίτηση της αρχιτεκτονικής Transformer.
torch.set_num_threads(1)

import torch.nn as nn
from torch.nn import functional as F

# ==============================================================================
# 1. ΥΠΕΡΠΑΡΑΜΕΤΡΟΙ
# ==============================================================================
# Το μοντέλο είναι character-level: κάθε token είναι ΕΝΑ χαρακτήρας και όχι λέξη/subword.
block_size = 32         # Μέγιστο context: έως 32 προηγούμενοι χαρακτήρες.
n_embd = 32             # Κάθε token και κάθε θέση αναπαρίστανται από διάνυσμα 32 διαστάσεων.
n_head = 4              # 4 attention heads. Κάθε head θα έχει 32/4 = 8 διαστάσεις.
n_layer = 4             # 4 διαδοχικά Transformer blocks.
learning_rate = 1e-3    # Βήμα του optimizer AdamW.
max_iters = 1500        # Πόσα gradient-update βήματα θα εκτελέσουμε.
eval_interval = 300     # Κάθε πόσα βήματα τυπώνουμε το loss του τρέχοντος batch.

device = 'cpu'          # Σκόπιμα CPU: το μοντέλο είναι αρκετά μικρό για το εκπαιδευτικό πείραμα.
torch.manual_seed(1337) # Σταθερό seed ώστε οι τυχαίες αρχικοποιήσεις/sampling να είναι αναπαραγώγιμα.

# ==============================================================================
# 2. DATASET
# ==============================================================================
# Μικρό συνθετικό corpus. Το μέγεθός του είναι εσκεμμένα μικρό ώστε να φαίνεται καθαρά
# η λειτουργία ολόκληρου του pipeline: tokenization -> batches -> training -> generation.
text = """Μια φορά κι έναν καιρό ήταν ένας σοφός βασιλιάς.
Ο βασιλιάς ζούσε σε ένα μεγάλο πέτρινο κάστρο.
Μια φορά κι έναν καιρό ήταν μια καλή βασίλισσα.
Η βασίλισσα αγαπούσε όλα τα ζώα του δάσους.
Υπήρχε κάποτε ένας νεαρός πρίγκιπας στο κάστρο.
Ο πρίγκιπας είχε ένα μικρό άσπρο άλογο.
Υπήρχε κάποτε μια όμορφη πριγκίπισσα.
Η πριγκίπισσα διάβαζε παλιά βιβλία στη βιβλιοθήκη.
Ο βασιλιάς και η βασίλισσα κυβερνούσαν με δικαιοσύνη.
Το κάστρο ήταν χτισμένο πάνω σε έναν ψηλό λόφο.
Ένα μικρό κορίτσι επισκέφτηκε το παλάτι μια μέρα.
Ο πρίγκιπας και η πριγκίπισσα έπαιζαν συχνά στον κήπο.
"""

# ==============================================================================
# 3. CHARACTER TOKENIZER
# ==============================================================================
# Βρίσκουμε όλους τους διαφορετικούς χαρακτήρες του corpus και τους ταξινομούμε.
# Το vocab_size είναι επομένως ο αριθμός των διαφορετικών χαρακτήρων που μπορεί να προβλέψει το μοντέλο.
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Δύο αντίστροφα λεξικά: χαρακτήρας -> integer token id και token id -> χαρακτήρας.
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

# encode: κείμενο -> λίστα ακεραίων. decode: λίστα ακεραίων -> κείμενο.
encode = lambda s: [char_to_id[c] for c in s]
decode = lambda l: ''.join([id_to_char[i] for i in l])

# Όλο το corpus γίνεται ένα μονοδιάστατο tensor από token IDs.
# dtype=torch.long επειδή τα nn.Embedding περιμένουν ακέραιους δείκτες.
data = torch.tensor(encode(text), dtype=torch.long)

def get_batch():
    # Παίρνουμε 16 τυχαία σημεία εκκίνησης από το corpus.
    # Το 16 είναι το batch size αυτού του πειράματος.
    ix = torch.randint(len(data) - block_size, (16,))

    # x: 16 ακολουθίες των 32 χαρακτήρων, shape = (B=16, T=32).
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)

    # y: η ίδια ακολουθία μετατοπισμένη κατά ΕΝΑ χαρακτήρα.
    # Άρα για κάθε θέση στο x, target είναι ο επόμενος χαρακτήρας.
    # Αυτό ακριβώς είναι next-token prediction.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

# ==============================================================================
# 4. TRANSFORMER
# ==============================================================================
class Head(nn.Module):
    """Μία causal self-attention head."""
    def __init__(self, head_size):
        super().__init__()
        # Από κάθε embedding παράγουμε Key, Query και Value.
        # Δεν χρησιμοποιούμε bias για να κρατήσουμε την υλοποίηση μικρή και καθαρή.
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Κάτω τριγωνική μάσκα. Θέση t επιτρέπεται να κοιτάξει μόνο θέσεις <= t,
        # ποτέ μελλοντικούς χαρακτήρες. register_buffer σημαίνει ότι αποθηκεύεται
        # μαζί με το module αλλά ΔΕΝ είναι εκπαιδεύσιμη παράμετρος.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # B=batch, T=μήκος ακολουθίας, C=embedding dimension (=32 εδώ).
        B, T, C = x.shape
        k = self.key(x)    # (B,T,head_size)
        q = self.query(x)  # (B,T,head_size)

        # Dot-product attention scores ανάμεσα σε όλα τα ζεύγη θέσεων.
        # Η κλιμάκωση αποτρέπει τα dot products από το να γίνονται υπερβολικά μεγάλα.
        wei = q @ k.transpose(-2, -1) * (C**-0.5)  # (B,T,T)

        # Οι μελλοντικές θέσεις παίρνουν -inf, άρα μετά το softmax πιθανότητα 0.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)  # κάθε γραμμή γίνεται κατανομή βαρών attention

        v = self.value(x)             # πληροφορία που τελικά θα συγκεντρωθεί
        out = wei @ v                 # σταθμισμένο άθροισμα Values, shape (B,T,head_size)
        return out

class MultiHeadAttention(nn.Module):
    """Πολλαπλές attention heads που εξετάζουν την ίδια ακολουθία παράλληλα."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # Τα outputs των 4 heads ενώνονται: 4*8 = 32 dimensions.
        # Η projection τα αναμειγνύει ξανά στον κοινό embedding space.
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """Μικρό MLP που εφαρμόζεται ανεξάρτητα σε κάθε θέση της ακολουθίας."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Προσωρινή επέκταση 32 -> 128 dimensions.
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            # Επιστροφή 128 -> 32 ώστε να συνεχίσει το residual stream.
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Ένα Transformer block: attention + feed-forward, με pre-norm residual connections."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head  # 32 // 4 = 8 dimensions ανά head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual connection: κρατάμε το υπάρχον x και προσθέτουμε την πληροφορία του attention.
        x = x + self.sa(self.ln1(x))
        # Δεύτερο residual branch για το feed-forward δίκτυο.
        x = x + self.ffwd(self.ln2(x))
        return x

# ==============================================================================
# 5. ΠΛΗΡΕΣ ΓΛΩΣΣΙΚΟ ΜΟΝΤΕΛΟ
# ==============================================================================
class MiniLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Token embedding: ποιος χαρακτήρας είναι αυτός;
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Position embedding: σε ποια θέση 0..31 βρίσκεται;
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Στοιβάζουμε 4 ίδιας δομής Transformer blocks με ανεξάρτητα weights.
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)

        # Language-model head: για κάθε θέση μετατρέπει το embedding σε ένα logit
        # για ΚΑΘΕ πιθανό χαρακτήρα του vocabulary.
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # (B,T) -> (B,T,32)
        tok_emb = self.token_embedding_table(idx)
        # (T,) -> (T,32). Προστίθεται σε κάθε sequence του batch μέσω broadcasting.
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B,T,vocab_size)

        if targets is None:
            # Inference: δεν έχουμε σωστές απαντήσεις, άρα δεν χρειάζεται loss.
            loss = None
        else:
            # Η cross_entropy περιμένει predictions (N,C) και targets (N).
            # Ενώνουμε λοιπόν batch και time: (B,T,C) -> (B*T,C).
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        # Autoregressive generation: προβλέπουμε ΕΝΑ token, το προσθέτουμε,
        # και ξανατρέχουμε το μοντέλο για το επόμενο.
        for _ in range(max_new_tokens):
            # Το μοντέλο δεν μπορεί να δει πάνω από block_size=32 tokens.
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)

            # Μας ενδιαφέρει μόνο η πρόβλεψη της ΤΕΛΕΥΤΑΙΑΣ θέσης,
            # δηλαδή η κατανομή για τον επόμενο χαρακτήρα.
            logits = logits[:, -1, :] / temperature

            # Temperature < 1: οξύνει την κατανομή, πιο συντηρητικό output.
            # Temperature > 1: την εξομαλύνει, περισσότερη τυχαιότητα.
            probs = F.softmax(logits, dim=-1)

            # Sampling και όχι argmax: επιλέγουμε χαρακτήρα σύμφωνα με τις πιθανότητες.
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Δημιουργούμε το μοντέλο και μετράμε ΟΛΑ τα trainable tensors του.
model = MiniLanguageModel().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Το μοντέλο δημιουργήθηκε επιτυχώς! Συνολικές παράμετροι: {param_count:,}")

# ==============================================================================
# 6. TRAINING LOOP
# ==============================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\n--- Έναρξη Εκπαίδευσης ---")
for iter in range(max_iters):
    # Νέο τυχαίο mini-batch σε κάθε training step.
    xb, yb = get_batch()

    # Forward pass: predictions και cross-entropy loss.
    logits, loss = model(xb, yb)

    # Μηδενίζουμε gradients του προηγούμενου βήματος.
    optimizer.zero_grad(set_to_none=True)
    # Backpropagation: υπολογισμός gradients για όλες τις παραμέτρους.
    loss.backward()
    # AdamW update: μικρή μετακίνηση των weights προς κατεύθυνση που μειώνει το loss.
    optimizer.step()

    # Αυτό είναι training-batch loss, όχι ξεχωριστό validation estimate.
    if iter % eval_interval == 0 or iter == max_iters - 1:
        print(f"Βήμα {iter:4d} | Σφάλμα (Loss): {loss.item():.4f}")

# ==============================================================================
# 7. INFERENCE
# ==============================================================================
print("\n--- Δοκιμή Παραγωγής Κειμένου ---")
prompt = "Μια φορά "
# unsqueeze(0): προσθέτουμε batch dimension, shape (T,) -> (1,T).
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

generated_ids = model.generate(context, max_new_tokens=150, temperature=0.7)
print(decode(generated_ids[0].tolist()))


In [ ]:
# ==============================================================================
# 8. ΠΕΙΡΑΜΑ ΜΕ ΤΗ ΘΕΡΜΟΚΡΑΣΙΑ (TEMPERATURE)
# ==============================================================================
# Κρατάμε ΙΔΙΟ prompt και ΙΔΙΟ εκπαιδευμένο μοντέλο. Αλλάζουμε μόνο το temperature,
# ώστε να απομονώσουμε την επίδρασή του στη διαδικασία sampling.
prompt = "Υπήρχε κάποτε "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

print("--- Temperature 0.2 (Συντηρητικό) ---")
# Με T=0.2 οι διαφορές μεταξύ logits μεγεθύνονται πριν το softmax.
# Έτσι οι ήδη πιθανότεροι χαρακτήρες γίνονται ακόμη πιθανότεροι.
out_low = model.generate(context, max_new_tokens=120, temperature=0.2)
print(decode(out_low[0].tolist()))

print("\n--- Temperature 1.8 (Υπερβολικά τυχαίο / ασυναρτησία) ---")
# Με T=1.8 οι διαφορές μεταξύ logits μικραίνουν.
# Περισσότεροι χαρακτήρες αποκτούν ουσιαστική πιθανότητα επιλογής,
# άρα αυξάνεται η ποικιλία αλλά και η πιθανότητα ασυνάρτητων ακολουθιών.
out_high = model.generate(context, max_new_tokens=120, temperature=1.8)
print(decode(out_high[0].tolist()))
